# About PyTorch

For these models' implementation PyTorch will be used alongside its common structure. For information on in-code structural decisions, check: 
https://docs.pytorch.org/docs/2.12/generated/torch.nn.RNN.html
https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html
https://docs.pytorch.org/docs/2.12/generated/torch.nn.Module.html

# Code
## Libraries

In [35]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [36]:
data = pd.read_csv("../data/Load/data_sensor_D.csv")
data.head()
 

,periodo,datetime,CO2_ppm,AQI,CO_ppm,SO2_ppm,O3_ppm,NO2_ppm,PM2_5_ugm3,temp_C,hum_pct,Bat_pct
0,P1,2025-11-06 20:09:35,0.997565,5,0.446300,0.930365,-0.340872,-0.584938,0.443770,0.340532,0.434159,0.704261
1,P1,2025-11-06 20:10:18,0.837793,5,0.358256,1.085778,-0.113082,-0.303829,0.380526,0.338870,0.435463,0.700501
2,P1,2025-11-06 20:11:19,0.651854,4,0.311917,1.085778,-0.204198,-0.491235,0.380526,0.337209,0.432855,0.701754
3,P1,2025-11-06 20:12:19,0.530929,4,0.324274,1.023613,-0.158640,-0.444384,0.317281,0.337209,0.438070,0.703008
4,P1,2025-11-06 20:13:20,0.449070,4,0.284113,1.054696,-0.249756,-0.444384,0.317281,0.332226,0.436767,0.704261


In [37]:

numeric_data = data.drop(columns=["periodo", "datetime"]).copy()
X = numeric_data.drop(columns=["AQI"]).to_numpy()
y = numeric_data["AQI"].to_numpy()
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Transform and Reshape data

scaler_X = StandardScaler()
scaler_y = StandardScaler()
 
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)
 
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_sc  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

In [ ]:
def to_tensor_3d(X, y):
    """
    Function that turns data into tensors, such that it is compatible
    with PyTorch's RNN API
    """
    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)   # (N, 1, F)
    y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)   # (N, 1)
    return TensorDataset(X_t, y_t)

train_ds = to_tensor_3d(X_train_sc, y_train_sc)
test_ds  = to_tensor_3d(X_test_sc,  y_test_sc)
 
BATCH_SIZE = 64


# Considering no sliding window, assuming it is not sequential
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
 

Using device: cpu


In [ ]:
# Generalized, where to store computations? in case user has a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Summarized, to use for both models
INPUT_SIZE  = X_train.shape[1]
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
DROPOUT     = 0.2
EPOCHS      = 100
LR          = 1e-3

## Classes and functions
Generalized for use in other proyects

In [ ]:
class VanillaRNN(nn.Module):
    """Single-step regression with a vanilla (Elman) RNN."""
 
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            nonlinearity="tanh",
        )
        self.fc = nn.Linear(hidden_size, 1)
 
    def forward(self, x):
        out, _ = self.rnn(x) # out: (batch, seq, hidden)
        return self.fc(out[:, -1, :]) # last time-step → (batch, 1)

In [42]:
class GRUNet(nn.Module):
    """Single-step regression with a GRU."""
 
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)
 
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

In [ ]:
def train_model(model, loader, epochs=EPOCHS, lr=LR):
    """
    Function that manages and tracks training stage's epochs
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )
 
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(X_batch)
 
        avg_loss = epoch_loss / len(loader.dataset)
        scheduler.step(avg_loss)
 
        if epoch % 10 == 0:
            print(f"  Epoch {epoch:3d}/{epochs}  loss={avg_loss:.6f}")
 
    return model

In [ ]:
def evaluate_model(model, loader, scaler_y):
    """
    Function that predicts and perfurms evaluation metrics for models
    """
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            out = model(X_batch).cpu().numpy()
            preds.append(out)
            targets.append(y_batch.numpy())
 
    preds   = scaler_y.inverse_transform(np.vstack(preds)).ravel()
    targets = scaler_y.inverse_transform(np.vstack(targets)).ravel()
 
    rmse = np.sqrt(mean_squared_error(targets, preds))
    mae  = mean_absolute_error(targets, preds)
    return rmse, mae, preds, targets

In [ ]:
# --------------------------------
#           Vanilla RNN
# --------------------------------


print("\n── Vanilla RNN ──────────────────────────────────────────────────────")
rnn_model = VanillaRNN(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
rnn_model = train_model(rnn_model, train_loader)
 
rmse_rnn, mae_rnn, y_pred_rnn, _ = evaluate_model(rnn_model, test_loader, scaler_y)
print(f"Vanilla RNN RMSE: {rmse_rnn:.4f}")
print(f"Vanilla RNN MAE:  {mae_rnn:.4f}")
 

# --------------------------------
#               GRU
# --------------------------------
print("\n── GRU ──────────────────────────────────────────────────────────────")
gru_model = GRUNet(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
gru_model = train_model(gru_model, train_loader)
 
rmse_gru, mae_gru, y_pred_gru, y_true = evaluate_model(gru_model, test_loader, scaler_y)
print(f"GRU RMSE: {rmse_gru:.4f}")
print(f"GRU MAE:  {mae_gru:.4f}")
 



── Vanilla RNN ──────────────────────────────────────────────────────
  Epoch  10/100  loss=0.101352
  Epoch  20/100  loss=0.088674
  Epoch  30/100  loss=0.076680
  Epoch  40/100  loss=0.075041
  Epoch  50/100  loss=0.072097
  Epoch  60/100  loss=0.071385
  Epoch  70/100  loss=0.070995
  Epoch  80/100  loss=0.069846
  Epoch  90/100  loss=0.068558
  Epoch 100/100  loss=0.068279
Vanilla RNN RMSE: 0.2418
Vanilla RNN MAE:  0.1766

── GRU ──────────────────────────────────────────────────────────────
  Epoch  10/100  loss=0.098153
  Epoch  20/100  loss=0.095167
  Epoch  30/100  loss=0.093890
  Epoch  40/100  loss=0.093615
  Epoch  50/100  loss=0.089444
  Epoch  60/100  loss=0.066109
  Epoch  70/100  loss=0.054086
  Epoch  80/100  loss=0.044916
  Epoch  90/100  loss=0.041653
  Epoch 100/100  loss=0.039010
GRU RMSE: 0.1942
GRU MAE:  0.0875

── Results summary ──────────────────────────────────────────────────
      Model     RMSE      MAE
Vanilla RNN 0.241802 0.176560
        GRU 0.194167 0.

In [ ]:
# --------------------------------
#           Summary Metrics
# --------------------------------
print("\n── Results summary ──────────────────────────────────────────────────")
results = pd.DataFrame({
    "Model": ["Vanilla RNN", "GRU"],
    "RMSE":  [rmse_rnn,     rmse_gru],
    "MAE":   [mae_rnn,      mae_gru],
})
print(results.to_string(index=False))